# 3장 — 임베딩 공간 탐험 🧭

웹 챕터의 지도를 코드로 다시 만듭니다 — 클러스터를 **숫자로** 확인하고, 그 유명한
**왕 − 남자 + 여자** 산수를 직접 계산합니다.

**이 노트북의 목표**
1. 클러스터: 무리 **안** 거리가 무리 **밖** 거리보다 확실히 짧음을 확인
2. 의미의 방향: 여자−남자 = 공주−왕자 = 여왕−왕 (평행 화살표의 정체)
3. 단어 유추: 왕 − 남자 + 여자의 최근접 이웃이 정말 여왕인지 검증

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch3')

## 1. 지도 준비 — 미리 만든 2D toy 임베딩

성별 방향이 (+2, 0)이 되도록 설계된 12단어 지도입니다 (실제는 모델이 수백 차원에서 배움 — 9장).

In [ ]:
# toy 임베딩 (단어 → 2D 좌표) — 성별 방향이 (+2, 0)이 되도록 설계
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 라벨 (Windows 기본 폰트)
plt.rcParams['axes.unicode_minus'] = False

E = {
    '남자': (2, 2), '여자': (4, 2), '왕자': (2, 4), '공주': (4, 4), '왕': (2, 6), '여왕': (4, 6),
    '강아지': (7.5, 1.2), '고양이': (8.3, 1.8), '토끼': (7.9, 0.6),
    '사과': (8.8, 6.2), '바나나': (9.4, 5.6), '포도': (9.1, 6.8),
}
E = {w: np.array(v, dtype=float) for w, v in E.items()}
GROUPS = {'사람': ['남자', '여자', '왕자', '공주', '왕', '여왕'],
          '동물': ['강아지', '고양이', '토끼'],
          '과일': ['사과', '바나나', '포도']}

plt.figure(figsize=(6, 4.2))
for (name, words), color in zip(GROUPS.items(), ['tab:blue', 'tab:orange', 'tab:green']):
    xs = [E[w][0] for w in words]; ys = [E[w][1] for w in words]
    plt.scatter(xs, ys, c=color, label=name)
    for w in words:
        plt.annotate(w, E[w], textcoords='offset points', xytext=(5, 3), fontsize=9)
plt.legend(); plt.title('toy embedding map'); plt.grid(alpha=0.3); plt.show()

## 2. 실험 ① — 클러스터를 숫자로

"모여 있다"를 어떻게 잴까요? **무리 안 평균 거리 vs 무리 밖 평균 거리**를 비교하면 됩니다.

In [ ]:
def avg_dist(pairs):
    return float(np.mean([np.linalg.norm(E[a] - E[b]) for a, b in pairs]))

words = list(E)
group_of = {w: g for g, ws in GROUPS.items() for w in ws}
all_pairs = [(a, b) for i, a in enumerate(words) for b in words[i + 1:]]
inner = [(a, b) for a, b in all_pairs if group_of[a] == group_of[b]]
outer = [(a, b) for a, b in all_pairs if group_of[a] != group_of[b]]

d_in, d_out = avg_dist(inner), avg_dist(outer)
print(f'무리 안 평균 거리: {d_in:.2f}')
print(f'무리 밖 평균 거리: {d_out:.2f}  ({d_out / d_in:.1f}배)')
assert d_out > d_in * 1.5, '클러스터의 정의 — 안은 가깝고 밖은 멀어야'
print('\n✔ 확인: 비슷한 뜻은 모여 있다 — 질문이 같은 클러스터에 떨어지니 의미 검색이 된다')

## 3. 실험 ② — 의미의 방향: 평행 화살표

"남성→여성"이라는 뜻의 변화 = 뺄셈 벡터. 세 쌍에서 같은 벡터가 나오는지 봅니다.

In [ ]:
for male, female in [('남자', '여자'), ('왕자', '공주'), ('왕', '여왕')]:
    arrow = E[female] - E[male]
    print(f'   {female} − {male} = {arrow}')
    assert np.allclose(arrow, [2, 0]), '성별 방향은 세 쌍 모두 (2, 0)이어야 — 평행 화살표'
print('\n✔ 확인: 뜻의 변화(남성→여성)가 일정한 방향 벡터 하나로 존재한다')

## 4. 실험 ③ — 의미의 산수: 왕 − 남자 + 여자

방향이 뜻이라면, 벡터 덧셈·뺄셈(1부 2장)으로 뜻을 **계산**할 수 있습니다.
관례대로 계산 지점의 **최근접 이웃**(입력 세 단어 제외)을 답으로 삼습니다.

In [ ]:
def analogy(a, minus, plus):
    """a − minus + plus 지점의 최근접 단어 (입력 단어 제외 — word analogy 관례)."""
    target = E[a] - E[minus] + E[plus]
    candidates = [w for w in E if w not in (a, minus, plus)]
    best = min(candidates, key=lambda w: np.linalg.norm(E[w] - target))
    return best, target

for a, m, p, expected in [('왕', '남자', '여자', '여왕'),
                          ('왕자', '남자', '여자', '공주'),
                          ('여왕', '여자', '남자', '왕')]:
    best, target = analogy(a, m, p)
    print(f'   {a} − {m} + {p} = {target} → 최근접: {best}')
    assert best == expected, f'{a}−{m}+{p} 는 {expected}여야'
print('\n✔ 확인: 벡터 산수가 뜻을 맞혔다 — 단어 유추(word analogy)')

# 현실 체크: 실제 임베딩에선 ≈(근사) — 계산 지점에 단어가 정확히 있진 않다.
# 그리고 실무 RAG는 단어가 아니라 문장·문서 임베딩을 쓴다(공간 감각은 동일).

## 5. 관찰 정리 + 🤖 AI 연결

- **클러스터**: 무리 안 거리 ≪ 무리 밖 거리 — 의미 검색이 작동하는 토대.
- **의미의 방향**: 뜻의 변화(성별 등)가 일정한 방향 벡터로 존재 — 위치만이 아니라 방향도 뜻이다.
- **단어 유추**(word analogy): 왕 − 남자 + 여자 ≈ 여왕 — 1부 2장의 벡터 덧셈·뺄셈이 뜻의 계산이 된다.
  word2vec(2013)이 이걸 해내며 임베딩 시대를 열었다. 단, 실제론 ≈(최근접 이웃)이고,
  훈련 텍스트의 고정관념도 방향으로 학습된다(**임베딩 편향** — 의사−남자+여자=간호사 사례).

**다음 챕터**: 4장 — 고차원의 저주 (핵심). 이 지도를 수백 차원으로 올리면 이상한 일이 벌어집니다:
아무 벡터나 두 개 뽑으면 **거의 직교**(cos ≈ 0). 그런데도 임베딩 검색이 작동하는 이유는?